# 05 — Improved Model: LightGBM (calibrated)

**Owner:** Person A (modelling track).

**Rubric line:** Improved model + comparison to baseline.

Trains a LightGBM classifier on the deployable feature set, tunes
hyperparameters with Optuna, and calibrates the output probabilities so
the decision framework can use them directly.


In [ ]:
# --- Setup --------------------------------------------------------------
# Make `src/` importable regardless of where the notebook is launched from.
import sys, pathlib
PROJECT_ROOT = pathlib.Path.cwd()
while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src import config, data, features, models, metrics, decision, viz

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)


## 5.1 — Load split + features

In [ ]:
train_df, test_df = data.load_interim()
y_train = train_df[config.TARGET_COL]
y_test = test_df[config.TARGET_COL]
feature_cols = data.feature_columns(include_leaky=False)
X_train = train_df[feature_cols]
X_test = test_df[feature_cols]


## 5.2 — Hyperparameter tuning with Optuna

Tune on cross-validated PR-AUC over the train set. Test set is untouched.


In [ ]:
import optuna
from sklearn.model_selection import cross_val_score, StratifiedKFold

def objective(trial):
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 127),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'scale_pos_weight': models.positive_class_weight(y_train),
    }
    pipe = models.make_lgbm_pipeline(params)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=config.RANDOM_SEED)
    # n_jobs=1 avoids a Windows Proactor event-loop hang inside Jupyter kernels
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='average_precision', n_jobs=1)
    return scores.mean()

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=config.N_TUNING_TRIALS, show_progress_bar=True)
print('Best CV PR-AUC:', study.best_value)
study.best_params

## 5.3 — Refit best model and calibrate

In [ ]:
best_params = {**study.best_params, 'scale_pos_weight': models.positive_class_weight(y_train)}
lgbm_pipe = models.make_lgbm_pipeline(best_params)
calibrated = models.calibrate(lgbm_pipe, X_train, y_train, method='isotonic')
lgbm_proba = calibrated.predict_proba(X_test)[:, 1]
lgbm_summary = metrics.classification_summary(y_test.values, lgbm_proba)
lgbm_summary


## 5.4 — Side-by-side: baseline vs. improved

In [ ]:
import joblib
baseline_artifact = joblib.load(config.MODELS_DIR / 'baseline_logreg.joblib')
logreg_proba = baseline_artifact['proba_test']
logreg_summary = metrics.classification_summary(y_test.values, logreg_proba)

comparison = pd.DataFrame(
    [logreg_summary, lgbm_summary],
    index=['Logistic baseline', 'LightGBM (calibrated)'],
)
comparison = comparison.rename(columns={'brier': 'brier_score'})
comparison.to_csv(config.TABLES_DIR / 'baseline_vs_improved.csv')
comparison

## 5.5 — Calibration check

In [ ]:
cal_table = metrics.calibration_table(y_test.values, lgbm_proba, n_bins=10)
fig, ax = plt.subplots()
viz.plot_calibration(cal_table, ax=ax)
viz.save_fig(fig, '05_calibration')
cal_table


## 5.6 — Persist improved model + test predictions

In [ ]:
joblib.dump(
    {'model': calibrated, 'proba_test': lgbm_proba, 'best_params': best_params},
    config.MODELS_DIR / 'improved_lgbm.joblib',
)


## 5.7 — Plain-language interpretation

LightGBM improved PR-AUC from **0.460 to 0.495** (+3.5 pp) and ROC-AUC from 0.801 to 0.817, suggesting the tree model is better at capturing non-linear interactions between macro-economic indicators (euribor3m, emp.var.rate) and contact-history features that logistic regression approximates linearly.

The calibration plot shows predicted probabilities closely tracking observed conversion rates across bins, confirmed by the Brier score dropping from 0.162 to 0.074 (54% improvement) — isotonic calibration has effectively corrected the overconfident raw LightGBM outputs, making the probabilities suitable for the decision framework's expected-value calculations.

The main red flag is that at the default 0.5 threshold, recall collapses to 26% (vs 65% for logistic baseline) because the calibrated probabilities cluster below 0.5 for most positive cases; this is expected behaviour and is resolved by threshold optimisation in notebook 07 — do not interpret the lower F1 at 0.5 as a model quality regression.